In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('company_esg_financial_dataset.csv')
df.head()

,CompanyID,CompanyName,Industry,Region,Year,Revenue,ProfitMargin,MarketCap,GrowthRate,ESG_Overall,ESG_Environmental,ESG_Social,ESG_Governance,CarbonEmissions,WaterUsage,EnergyConsumption
0,1,Company_1,Retail,Latin America,2015,459.2,6.0,337.5,NaN,57.0,60.7,33.5,76.8,35577.4,17788.7,71154.7
1,1,Company_1,Retail,Latin America,2016,473.8,4.6,366.6,3.2,56.7,58.9,32.8,78.5,37314.7,18657.4,74629.4
2,1,Company_1,Retail,Latin America,2017,564.9,5.2,313.4,19.2,56.5,57.6,34.0,77.8,45006.4,22503.2,90012.9
3,1,Company_1,Retail,Latin America,2018,558.4,4.3,283.0,-1.1,58.0,62.3,33.4,78.3,42650.1,21325.1,85300.2
4,1,Company_1,Retail,Latin America,2019,554.5,4.9,538.1,-0.7,56.6,63.7,30.0,76.1,41799.4,20899.7,83598.8


In [3]:
df.isnull().sum()

CompanyID               0
CompanyName             0
Industry                0
Region                  0
Year                    0
Revenue                 0
ProfitMargin            0
MarketCap               0
GrowthRate           1000
ESG_Overall             0
ESG_Environmental       0
ESG_Social              0
ESG_Governance          0
CarbonEmissions         0
WaterUsage              0
EnergyConsumption       0
dtype: int64

In [4]:
details=['CompanyID', 'CompanyName', 'Industry']
financial=['Revenue', 'ProfitMargin', 'MarketCap','GrowthRate']
esg=['ESG_Overall', 'ESG_Environmental', 'ESG_Social', 'ESG_Governance', 'CarbonEmissions', 'WaterUsage', 'EnergyConsumption']

In [5]:
print(df.columns.tolist())
print(df.isna().sum())
print(df[['Year','GrowthRate','ProfitMargin','Revenue','MarketCap','CarbonEmissions','WaterUsage','EnergyConsumption']].head())
print('shape', df.shape)

['CompanyID', 'CompanyName', 'Industry', 'Region', 'Year', 'Revenue', 'ProfitMargin', 'MarketCap', 'GrowthRate', 'ESG_Overall', 'ESG_Environmental', 'ESG_Social', 'ESG_Governance', 'CarbonEmissions', 'WaterUsage', 'EnergyConsumption']
CompanyID               0
CompanyName             0
Industry                0
Region                  0
Year                    0
Revenue                 0
ProfitMargin            0
MarketCap               0
GrowthRate           1000
ESG_Overall             0
ESG_Environmental       0
ESG_Social              0
ESG_Governance          0
CarbonEmissions         0
WaterUsage              0
EnergyConsumption       0
dtype: int64
   Year  GrowthRate  ProfitMargin  Revenue  MarketCap  CarbonEmissions  \
0  2015         NaN           6.0    459.2      337.5          35577.4   
1  2016         3.2           4.6    473.8      366.6          37314.7   
2  2017        19.2           5.2    564.9      313.4          45006.4   
3  2018        -1.1           4.3    558

In [6]:
print(f"    Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"    Companies: {df['CompanyID'].nunique()} | Years: {df['Year'].min()}–{df['Year'].max()}")
print(f"    Industries: {df['Industry'].nunique()} | Regions: {df['Region'].nunique()}")
print(f"    Missing: GrowthRate={df['GrowthRate'].isna().sum()} rows")

    Shape: 11,000 rows × 16 columns
    Companies: 1000 | Years: 2015–2025
    Industries: 9 | Regions: 7
    Missing: GrowthRate=1000 rows


``` Step-2 feature engineering```

In [7]:

# 2a. Environmental Intensity Features (KEY DRIVER for E-score)
df['carbon_intensity']  = df['CarbonEmissions']   / (df['Revenue'] + 1e-6)
df['water_intensity']   = df['WaterUsage']         / (df['Revenue'] + 1e-6)
df['energy_intensity']  = df['EnergyConsumption']  / (df['Revenue'] + 1e-6)

In [8]:
df.columns.tolist()

['CompanyID',
 'CompanyName',
 'Industry',
 'Region',
 'Year',
 'Revenue',
 'ProfitMargin',
 'MarketCap',
 'GrowthRate',
 'ESG_Overall',
 'ESG_Environmental',
 'ESG_Social',
 'ESG_Governance',
 'CarbonEmissions',
 'WaterUsage',
 'EnergyConsumption',
 'carbon_intensity',
 'water_intensity',
 'energy_intensity']

In [9]:
# 2b. Log-transform skewed raw metrics
df['log_carbon']   = np.log1p(df['CarbonEmissions'])
df['log_water']    = np.log1p(df['WaterUsage'])
df['log_energy']   = np.log1p(df['EnergyConsumption'])
df['log_revenue']  = np.log1p(df['Revenue'])
df['log_marketcap']= np.log1p(df['MarketCap'])


In [10]:
# 2c. Composite environmental burden (normalized sum of intensities)
df['env_burden'] = (
    df['carbon_intensity'] + df['water_intensity'] + df['energy_intensity'])

In [11]:
# 2e. Year trend (normalized)
df['year_norm'] = (df['Year'] - df['Year'].min()) / (df['Year'].max() - df['Year'].min())

# 2f. Encode categoricals
le_industry = LabelEncoder()
le_region   = LabelEncoder()
df['industry_enc'] = le_industry.fit_transform(df['Industry'])
df['region_enc']   = le_region.fit_transform(df['Region'])

```STEP 3: DEFINE FEATURE SETS PER TARGET```

In [12]:
# Environmental features — intensity metrics are the primary drivers
env_features = [
    'carbon_intensity', 'water_intensity', 'energy_intensity',
    'env_burden',
    'log_carbon', 'log_water', 'log_energy',
    'log_revenue', 'log_marketcap',
    'ProfitMargin', 'GrowthRate',
    'industry_enc', 'region_enc', 'year_norm'
]

In [13]:
# Social features — industry/region/time trends dominate; financial proxy weak
social_features = [
    'industry_enc', 'region_enc', 'year_norm',
    'log_revenue', 'log_marketcap',
    'ProfitMargin', 'GrowthRate'
]

In [14]:
# Governance features — similar to social; org/structural factors dominate
gov_features = [
    'industry_enc', 'region_enc', 'year_norm',
    'log_revenue', 'log_marketcap',
    'ProfitMargin', 'GrowthRate'
]

In [15]:
# Impute missing values in the selected model inputs before training
model_features = sorted(set(env_features + social_features + gov_features))
imputer = SimpleImputer(strategy='median')
df[model_features] = imputer.fit_transform(df[model_features])

df = df.dropna(subset=['ESG_Environmental', 'ESG_Social', 'ESG_Governance'])
print(f"    Imputed missing values in {len(model_features)} model features; remaining rows: {len(df):,}")
print("    Environmental features:", len(env_features))
print("    Social features:", len(social_features))
print("    Governance features:", len(gov_features))


    Imputed missing values in 14 model features; remaining rows: 11,000
    Environmental features: 14
    Social features: 7
    Governance features: 7


```STEP 4: TRAIN/TEST SPLIT```

In [16]:
# Temporal split: train on 2015–2022, test on 2023–2025
train_df = df[df['Year'] <= 2022].copy()
test_df  = df[df['Year'] > 2022].copy()
print(f"    Train: {len(train_df):,} rows (2015–2022)")
print(f"    Test : {len(test_df):,} rows (2023–2025)")

    Train: 8,000 rows (2015–2022)
    Test : 3,000 rows (2023–2025)


In [17]:
# Fallback: random split if temporal gives too little test data
if len(test_df) < 500:
    print("    → Falling back to random 80/20 split")
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

```STEP 5: MODEL TRAINING — ONE MODEL PER ESG PILLAR```

In [18]:
targets = {
    'ESG_Environmental': (env_features, 'Environmental (E)'),
    'ESG_Social':        (social_features, 'Social (S)'),
    'ESG_Governance':    (gov_features, 'Governance (G)'),
}

In [19]:
models = {}
results = {}

model_template = GradientBoostingRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)

In [20]:
for target, (features, label) in targets.items():
    print(f"\n    ▶ Predicting {label}...")

    X_train = train_df[features]
    y_train = train_df[target]
    X_test  = test_df[features]
    y_test  = test_df[target]

    model = GradientBoostingRegressor(
        n_estimators=300, max_depth=5,
        learning_rate=0.05, subsample=0.8, random_state=42
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_pred = np.clip(y_pred, 0, 100)   # ESG scores are bounded 0–100
    # Save model
    models[target] = model

# Save predictions and actual values
    results[target] = {
    'label': label,
    'y_test': y_test,
    'y_pred': y_pred,

    'R2': r2_score(y_test, y_pred),
    'MAE': mean_absolute_error(y_test, y_pred),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred))
    }
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    models[target]  = model
    results[target] = {
        'MAE': mae, 'RMSE': rmse, 'R2': r2,
        'y_test': y_test, 'y_pred': y_pred,
        'features': features, 'label': label
    }

    print(f"      R²={r2:.4f} | MAE={mae:.2f} | RMSE={rmse:.2f}")



    ▶ Predicting Environmental (E)...
      R²=0.9998 | MAE=0.21 | RMSE=0.40

    ▶ Predicting Social (S)...
      R²=0.3553 | MAE=15.24 | RMSE=18.82

    ▶ Predicting Governance (G)...
      R²=0.4354 | MAE=15.54 | RMSE=19.11


```STEP 6: PREDICT ESG_OVERALL```

In [21]:
print("\n" + "=" * 70)
print("  MODEL PERFORMANCE SUMMARY")
print("=" * 70)
print(f"  {'Target':<25} {'R²':>8} {'MAE':>8} {'RMSE':>8}")
print("  " + "-" * 52)
for key, res in results.items():
    print(f"  {res['label']:<25} {res['R2']:>8.4f} {res['MAE']:>8.2f} {res['RMSE']:>8.2f}")
print("=" * 70)


  MODEL PERFORMANCE SUMMARY
  Target                          R²      MAE     RMSE
  ----------------------------------------------------
  Environmental (E)           0.9998     0.21     0.40
  Social (S)                  0.3553    15.24    18.82
  Governance (G)              0.4354    15.54    19.11


In [22]:
# =====================================
# OVERALL ESG PREDICTION
# =====================================

env_pred = results['ESG_Environmental']['y_pred']
social_pred = results['ESG_Social']['y_pred']
gov_pred = results['ESG_Governance']['y_pred']

overall_esg = (
    env_pred +
    social_pred +
    gov_pred
) / 3

overall_results = pd.DataFrame({

    'Environmental_Pred': env_pred,
    'Social_Pred': social_pred,
    'Governance_Pred': gov_pred,
    'Overall_ESG_Pred': overall_esg

})

print(overall_results.head())

   Environmental_Pred  Social_Pred  Governance_Pred  Overall_ESG_Pred
0           68.427641    52.929416        49.899155         57.085404
1           68.957911    50.477978        49.724372         56.386754
2           70.515289    53.353622        49.998867         57.955926
3           66.490831    45.819166        67.715226         60.008408
4           65.442286    52.277769        66.845820         61.521958


In [23]:
import joblib

joblib.dump(
    models['ESG_Environmental'],
    'env_model.pkl'
)

joblib.dump(
    models['ESG_Social'],
    'social_model.pkl'
)

joblib.dump(
    models['ESG_Governance'],
    'gov_model.pkl'
)

print("All models saved.")

All models saved.


```STEP 10: SCORE A NEW COMPANY (INFERENCE DEMO)```

In [24]:
models['ESG_Environmental']

,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.05
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",300
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",0.8
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are""friedman_mse"" for the mean squared error with improvement score byFriedman, ""squared_error"" for mean squared error. The default value of""friedman_mse"" is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",5
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft

In [25]:
new_company = pd.DataFrame({

    'Revenue': [5000000],
    'ProfitMargin': [15],
    'MarketCap': [20000000],
    'GrowthRate': [12],

    'CarbonEmissions': [300],
    'WaterUsage': [500],
    'EnergyConsumption': [700],

    # Encoded categorical values
    'industry_enc': [2],
    'region_enc': [1],

    # Year normalization input
    'year_norm': [0.9]

})

In [26]:
# ==================================
# LOG FEATURES
# ==================================

new_company['log_carbon'] = np.log1p(
    new_company['CarbonEmissions']
)

new_company['log_water'] = np.log1p(
    new_company['WaterUsage']
)

new_company['log_energy'] = np.log1p(
    new_company['EnergyConsumption']
)

new_company['log_marketcap'] = np.log1p(
    new_company['MarketCap']
)

In [27]:
# ==================================
# INTENSITY FEATURES
# ==================================

new_company['carbon_intensity'] = (
    new_company['CarbonEmissions']
    / new_company['Revenue']
)

new_company['water_intensity'] = (
    new_company['WaterUsage']
    / new_company['Revenue']
)

new_company['energy_intensity'] = (
    new_company['EnergyConsumption']
    / new_company['Revenue']
)

# ==================================
# ENVIRONMENTAL BURDEN
# ==================================

new_company['env_burden'] = (
    new_company['carbon_intensity']
    + new_company['water_intensity']
    + new_company['energy_intensity']
)

# ==================================
# LOG REVENUE
# ==================================

new_company['log_revenue'] = np.log1p(
    new_company['Revenue']
)

In [28]:
X_env_new = new_company[env_features]
X_social_new = new_company[social_features]
X_gov_new = new_company[gov_features]

# ==================================
# PREDICTIONS
# ==================================

env_score = models['ESG_Environmental'].predict(X_env_new)[0]

social_score = models['ESG_Social'].predict(X_social_new)[0]

gov_score = models['ESG_Governance'].predict(X_gov_new)[0]

overall_esg = (
    env_score +
    social_score +
    gov_score
) / 3

# ==================================
# PRINT RESULTS
# ==================================

print("\n===== ESG PREDICTION =====")

print(f"Environmental Score : {env_score:.2f}")
print(f"Social Score        : {social_score:.2f}")
print(f"Governance Score    : {gov_score:.2f}")
print(f"Overall ESG Score   : {overall_esg:.2f}")


===== ESG PREDICTION =====
Environmental Score : 99.99
Social Score        : 58.18
Governance Score    : 42.07
Overall ESG Score   : 66.75


In [29]:
X_env_new = new_company[env_features]
X_social_new = new_company[social_features]
X_gov_new = new_company[gov_features]

# ==================================
# PREDICTIONS
# ==================================

env_score = models['ESG_Environmental'].predict(X_env_new)[0]

social_score = models['ESG_Social'].predict(X_social_new)[0]

gov_score = models['ESG_Governance'].predict(X_gov_new)[0]

overall_esg = (
    0.4 * env_score +
    0.3 * social_score +
    0.3 * gov_score
)

# ==================================
# PRINT RESULTS
# ==================================
print("\n===== ESG PREDICTION =====")

print(f"Environmental Score : {env_score:.2f}")
print(f"Social Score        : {social_score:.2f}")
print(f"Governance Score    : {gov_score:.2f}")
print(f"Overall ESG Score   : {overall_esg:.2f}")


===== ESG PREDICTION =====
Environmental Score : 99.99
Social Score        : 58.18
Governance Score    : 42.07
Overall ESG Score   : 70.07


In [30]:
import joblib

joblib.dump(env_features, 'env_features.pkl')
joblib.dump(social_features, 'social_features.pkl')
joblib.dump(gov_features, 'gov_features.pkl')

['gov_features.pkl']